# NBA DFS Season Backtest - Per Player Models with Benchmark Comparison

Walk-forward backtesting across multiple slates using per-player XGBoost models and season average benchmark comparison.

## Methodology

1. Load historical data for training
2. Load injury data per slate for injury status features
3. Build features using YAML-configured rolling statistics and injury features
4. Train separate XGBoost model per player on historical data (with optional GPU acceleration)
5. Calculate season average benchmark for comparison
6. Walk forward through test period, generating predictions for each slate
7. Compare predictions to actual results
8. Analyze model vs benchmark performance overall and by salary tier
9. Statistical significance testing across all slates

## GPU Acceleration

This notebook supports GPU-accelerated training with XGBoost 2.0+:
- Set `USE_GPU = True` in the configuration cell to enable GPU training
- Optionally specify `MODEL_CONFIG_PATH` to load optimized hyperparameters from YAML
- XGBoost will automatically use CUDA for faster model training
- Recommended for large-scale backtests with many per-player models

GPU configuration example:
```python
USE_GPU = True
GPU_ID = 0  # First GPU
MODEL_CONFIG_PATH = 'config/models/xgboost_a100.yaml'  # Optional: GPU-tuned params
```

## Injury Data Integration

Injury data is automatically loaded per slate and merged with player features:
- **injury_status**: Healthy, Out, Questionable, Doubtful, Day-To-Day
- **is_injured**: Binary flag (1 if any injury designation)
- **is_out**: Binary flag (1 if ruled out)
- **is_questionable**: Binary flag (1 if questionable)
- **is_doubtful**: Binary flag (1 if doubtful)
- **is_day_to_day**: Binary flag (1 if day-to-day)
- **injury_designation**: Original injury designation text
- **injury_description**: Injury details from API

Injury features are configured in `config/features/default_features.yaml` and `config/features/base_features.yaml` via the `InjuryTransformer`.

## Player Filtering

Filter players before model training and projection. Multiple filter types can be combined (AND logic):

### Salary Filtering
- **FILTER_SALARY_MIN**: Set to int to include only players above this salary (e.g., 5000 for $5k+)
- **FILTER_SALARY_MAX**: Set to int to include only players below this salary (e.g., 10000 for up to $10k)

### Injury Filtering
- **FILTER_EXCLUDE_OUT**: Set to True to exclude OUT players
- **FILTER_EXCLUDE_DOUBTFUL**: Set to True to exclude DOUBTFUL players
- **FILTER_EXCLUDE_QUESTIONABLE**: Set to True to exclude QUESTIONABLE players

### Player ID/Name Filtering
- **FILTER_PLAYER_IDS**: Filter by player IDs. Supports:
  - List format: `[201935, 2544, 201950]`
  - Comma-separated string: `"201935,2544,201950"`
  - Space-separated string: `"201935 2544 201950"`
  - Single ID: `201935`
  
- **FILTER_PLAYER_NAMES**: Filter by player names (partial match, case-insensitive). Supports:
  - List format: `['LeBron', 'Durant']`
  - Comma-separated string: `"LeBron,Durant"`
  - Space-separated string: `"LeBron Durant"`
  - Single name: `"LeBron"`

- **FILTER_PLAYERS_CSV**: Filter by player IDs from a CSV file. CSV must contain a 'playerID' column.
  - Example: `"my_players.csv"` or `"/path/to/players.csv"`

Filters apply to both training and projection phases.

## Setup

In [13]:
import sys
from pathlib import Path
import logging
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from scipy import stats

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.walk_forward_backtest import WalkForwardBacktest
from src.data.loaders.historical_loader import HistoricalDataLoader

pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print('Setup complete')

Setup complete


## Configuration

In [14]:
import psutil
import multiprocessing

cpu_count = multiprocessing.cpu_count()
ram_gb = psutil.virtual_memory().total / (1024**3)

print(f"CPU Cores: {cpu_count}")
print(f"RAM: {ram_gb:.1f} GB")
print(f"Recommended n_jobs: {cpu_count}")

if ram_gb < 12:
    print("WARNING: Low RAM detected. Consider reducing n_jobs or processing fewer players.")

CPU Cores: 32
RAM: 31.8 GB
Recommended n_jobs: 32


In [15]:
DB_PATH = str(repo_root / 'nba_dfs.db')
OUTPUT_DIR = str(repo_root / 'data' / 'outputs')

TEST_START = '20250205'
TEST_END = '20250215'

NUM_SEASONS = 1
FEATURE_CONFIG = 'opponent_features'
MODEL_TYPE = 'xgboost'
MIN_PLAYER_GAMES = 10
MIN_GAMES_FOR_BENCHMARK = 5
RECALIBRATE_DAYS = 7
SALARY_TIERS = [0, 4000, 6000, 8000, 15000]

PER_PLAYER_MODELS = False
SAVE_MODELS = True
SAVE_PREDICTIONS = True
N_JOBS = 32

# GPU configuration
USE_GPU = True  # Set to True to enable GPU acceleration
GPU_ID = 0  # GPU device ID (0 for first GPU)
MODEL_CONFIG_PATH = "C:\\Users\\antho\\OneDrive\\Documents\\Repositories\\delapan-fantasy\\config\\models\\xgboost_default.yaml"  
# Path to GPU-optimized model config (e.g., 'config/models/xgboost_a100.yaml')

# Player filtering configuration - Salary and Injury filters
FILTER_SALARY_MIN = 5000  # Set to int to filter players below this salary
FILTER_SALARY_MAX = None  # Set to int to filter players above this salary
FILTER_EXCLUDE_OUT = False  # Set to True to exclude OUT players
FILTER_EXCLUDE_DOUBTFUL = False  # Set to True to exclude DOUBTFUL players
FILTER_EXCLUDE_QUESTIONABLE = False  # Set to True to exclude QUESTIONABLE players

# Player filtering configuration - ID and Name filters
FILTER_PLAYER_IDS = None  # Set to list of player IDs, e.g., [201935, 2544] or comma/space-separated string
FILTER_PLAYER_NAMES = ['Lebron James', 'Stephen Curry']  # Set to list of player names, e.g., ['LeBron', 'Durant'] or comma/space-separated string
FILTER_PLAYERS_CSV = None  # Set to CSV file path with playerID column, e.g., 'my_players.csv'

TRAIN_START = HistoricalDataLoader.get_season_start_date(TEST_START) if NUM_SEASONS == 1 else HistoricalDataLoader.get_previous_season_start_date(TEST_START)
TEST_END_DT = datetime.strptime(TEST_END, '%Y%m%d')
TRAIN_END = (TEST_END_DT - timedelta(days=1)).strftime('%Y%m%d')

# Load GPU-optimized model config if specified, otherwise use default params
if USE_GPU and MODEL_CONFIG_PATH:
    import yaml
    with open(repo_root / MODEL_CONFIG_PATH, 'r') as f:
        gpu_config = yaml.safe_load(f)
        MODEL_PARAMS = gpu_config.get('model', {}).get('params', {})
        # Ensure GPU device is set
        if 'device' not in MODEL_PARAMS:
            MODEL_PARAMS['device'] = f'cuda:{GPU_ID}'
        if 'tree_method' not in MODEL_PARAMS:
            MODEL_PARAMS['tree_method'] = 'hist'
else:
    MODEL_PARAMS = {
        'max_depth': 6,
        'learning_rate': 0.05,
        'n_estimators': 200,
        'min_child_weight': 5,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'objective': 'reg:squarederror',
        'random_state': 42
    }
    # Add GPU params if GPU enabled but no config file
    if USE_GPU:
        MODEL_PARAMS['device'] = f'cuda:{GPU_ID}'
        MODEL_PARAMS['tree_method'] = 'hist'

print('Configuration:')
print(f'  Database: {DB_PATH}')
print(f'  Output Directory: {OUTPUT_DIR}')
print(f'  Training Period: {TRAIN_START} to {TRAIN_END}')
print(f'  Testing Period: {TEST_START} to {TEST_END}')
print(f'  Number of Seasons: {NUM_SEASONS}')
print(f'  Model Type: {MODEL_TYPE}')
print(f'  Feature Config: {FEATURE_CONFIG}')
print(f'  Per-Player Models: {PER_PLAYER_MODELS}')
print(f'  Min Player Games: {MIN_PLAYER_GAMES}')
print(f'  Min Benchmark Games: {MIN_GAMES_FOR_BENCHMARK}')
print(f'  Recalibrate Every: {RECALIBRATE_DAYS} days')
print(f'  Parallel Jobs: {N_JOBS} ({"all cores" if N_JOBS == -1 else "sequential" if N_JOBS == 1 else f"{N_JOBS} workers"})')
print(f'  Save Models: {SAVE_MODELS}')
print(f'  Save Predictions: {SAVE_PREDICTIONS}')
print(f'  Salary Tiers: {SALARY_TIERS}')

if USE_GPU:
    print(f'\n  GPU Configuration:')
    print(f'    Enabled: Yes')
    print(f'    GPU ID: {GPU_ID}')
    print(f'    Device: {MODEL_PARAMS.get("device", "N/A")}')
    print(f'    Tree Method: {MODEL_PARAMS.get("tree_method", "N/A")}')
    if MODEL_CONFIG_PATH:
        print(f'    Config File: {MODEL_CONFIG_PATH}')
else:
    print(f'\n  GPU Configuration: Disabled (CPU mode)')

print(f'\n  Player Filters:')
print(f'    Salary & Injury:')
if FILTER_SALARY_MIN:
    print(f'      - Minimum salary: ${FILTER_SALARY_MIN}')
if FILTER_SALARY_MAX:
    print(f'      - Maximum salary: ${FILTER_SALARY_MAX}')
if FILTER_EXCLUDE_OUT:
    print(f'      - Exclude OUT players')
if FILTER_EXCLUDE_DOUBTFUL:
    print(f'      - Exclude DOUBTFUL players')
if FILTER_EXCLUDE_QUESTIONABLE:
    print(f'      - Exclude QUESTIONABLE players')

print(f'\n    Player ID/Name:')
if FILTER_PLAYER_IDS:
    if isinstance(FILTER_PLAYER_IDS, str):
        ids_list = [id.strip() for id in FILTER_PLAYER_IDS.replace(',', ' ').split()]
    else:
        ids_list = FILTER_PLAYER_IDS if isinstance(FILTER_PLAYER_IDS, list) else [FILTER_PLAYER_IDS]
    ids_display = ', '.join(str(id) for id in ids_list[:3])
    if len(ids_list) > 3:
        ids_display += f", ... (+{len(ids_list)-3} more)"
    print(f'      - Filter by player IDs: {ids_display}')
if FILTER_PLAYER_NAMES:
    if isinstance(FILTER_PLAYER_NAMES, str):
        names_list = [name.strip() for name in FILTER_PLAYER_NAMES.replace(',', '|').split('|')]
    else:
        names_list = FILTER_PLAYER_NAMES if isinstance(FILTER_PLAYER_NAMES, list) else [FILTER_PLAYER_NAMES]
    names_display = ', '.join(names_list[:2])
    if len(names_list) > 2:
        names_display += f", ... (+{len(names_list)-2} more)"
    print(f'      - Filter by player names: {names_display}')
if FILTER_PLAYERS_CSV:
    print(f'      - Filter from CSV file: {FILTER_PLAYERS_CSV}')

if not any([FILTER_SALARY_MIN, FILTER_SALARY_MAX, FILTER_EXCLUDE_OUT, FILTER_EXCLUDE_DOUBTFUL, 
            FILTER_EXCLUDE_QUESTIONABLE, FILTER_PLAYER_IDS, FILTER_PLAYER_NAMES, FILTER_PLAYERS_CSV]):
    print(f'      - No player filters configured')

Configuration:
  Database: c:\Users\antho\OneDrive\Documents\Repositories\delapan-fantasy\nba_dfs.db
  Output Directory: c:\Users\antho\OneDrive\Documents\Repositories\delapan-fantasy\data\outputs
  Training Period: 20241001 to 20250214
  Testing Period: 20250205 to 20250215
  Number of Seasons: 1
  Model Type: xgboost
  Feature Config: opponent_features
  Per-Player Models: False
  Min Player Games: 10
  Min Benchmark Games: 5
  Recalibrate Every: 7 days
  Parallel Jobs: 32 (32 workers)
  Save Models: True
  Save Predictions: True
  Salary Tiers: [0, 4000, 6000, 8000, 15000]

  GPU Configuration:
    Enabled: Yes
    GPU ID: 0
    Device: cuda:0
    Tree Method: hist
    Config File: C:\Users\antho\OneDrive\Documents\Repositories\delapan-fantasy\config\models\xgboost_default.yaml

  Player Filters:
    Salary & Injury:
      - Minimum salary: $5000

    Player ID/Name:
      - Filter by player names: Lebron James, Stephen Curry


## Run Walk-Forward Backtest

Initialize and run the WalkForwardBacktest class. This will:
1. Load historical training data
2. Build features using YAML-configured pipeline
3. Initialize season average benchmark
4. Train per-player models (or slate-level model)
5. Generate predictions for each test slate
6. Evaluate against actuals
7. Perform statistical analysis

## Load Previous Results (Optional)

Skip this section if you want to run a new backtest.

Use this section to load results from a previous backtest run and jump directly to visualization/analysis.

In [16]:
# Configuration for loading previous results
LOAD_FROM_PREVIOUS = True  # Set to True to load from previous run
LOAD_RUN_TIMESTAMP = '20251019_191117'  # Specify timestamp like '20251019_050018' or None for most recent

if LOAD_FROM_PREVIOUS:
    from pathlib import Path
    from src.evaluation.metrics.accuracy import MAPEMetric, RMSEMetric, MAEMetric, CorrelationMetric
    
    output_path = Path(OUTPUT_DIR)
    run_dirs = sorted([d for d in output_path.iterdir() if d.is_dir()], reverse=True)
    
    if not run_dirs:
        print(f"ERROR: No previous runs found in {OUTPUT_DIR}")
    else:
        print(f"Found {len(run_dirs)} previous run(s)")
        
        # Select run
        selected_run = output_path / LOAD_RUN_TIMESTAMP if LOAD_RUN_TIMESTAMP else run_dirs[0]
        
        if not selected_run.exists():
            print(f"ERROR: Run {selected_run.name} not found")
        else:
            predictions_dir = selected_run / 'predictions'
            
            if not predictions_dir.exists():
                print(f"ERROR: No predictions directory in {selected_run.name}")
            else:
                print(f"Loading from: {selected_run.name}")
                
                # Load all slate results
                actuals_files = sorted(predictions_dir.glob('*_with_actuals.parquet'))
                
                if not actuals_files:
                    print("ERROR: No result files found")
                else:
                    all_predictions_list = [pd.read_parquet(f) for f in actuals_files]
                    all_predictions_df = pd.concat(all_predictions_list, ignore_index=True)
                    
                    # Reconstruct daily results
                    mape_metric, rmse_metric, mae_metric, corr_metric = MAPEMetric(), RMSEMetric(), MAEMetric(), CorrelationMetric()
                    
                    daily_results = []
                    for date in sorted(all_predictions_df['date'].unique()):
                        date_df = all_predictions_df[all_predictions_df['date'] == date]
                        has_benchmark = (date_df['benchmark_pred'] > 0)
                        
                        daily_results.append({
                            'date': date,
                            'num_players': len(date_df),
                            'model_mape': mape_metric.calculate(date_df['actual_fpts'], date_df['projected_fpts']),
                            'model_rmse': rmse_metric.calculate(date_df['actual_fpts'], date_df['projected_fpts']),
                            'model_mae': mae_metric.calculate(date_df['actual_fpts'], date_df['projected_fpts']),
                            'model_corr': corr_metric.calculate(date_df['actual_fpts'], date_df['projected_fpts']),
                            'benchmark_mape': mape_metric.calculate(date_df[has_benchmark]['actual_fpts'], date_df[has_benchmark]['benchmark_pred']) if has_benchmark.any() else np.nan,
                            'benchmark_rmse': rmse_metric.calculate(date_df[has_benchmark]['actual_fpts'], date_df[has_benchmark]['benchmark_pred']) if has_benchmark.any() else np.nan,
                            'mean_actual': date_df['actual_fpts'].mean(),
                            'mean_projected': date_df['projected_fpts'].mean(),
                            'mean_benchmark': date_df['benchmark_pred'].mean()
                        })
                    
                    results_df = pd.DataFrame(daily_results)
                    
                    # Create results dict
                    results = {
                        'num_slates': len(results_df),
                        'date_range': f"{results_df['date'].min()} to {results_df['date'].max()}",
                        'total_players_evaluated': results_df['num_players'].sum(),
                        'avg_players_per_slate': results_df['num_players'].mean(),
                        'model_mean_mape': results_df['model_mape'].mean(),
                        'model_median_mape': results_df['model_mape'].median(),
                        'model_std_mape': results_df['model_mape'].std(),
                        'model_mean_rmse': results_df['model_rmse'].mean(),
                        'model_mean_mae': results_df['model_mae'].mean(),
                        'model_mean_correlation': results_df['model_corr'].mean(),
                        'benchmark_mean_mape': results_df['benchmark_mape'].mean(),
                        'benchmark_median_mape': results_df['benchmark_mape'].median(),
                        'mape_improvement': results_df['benchmark_mape'].mean() - results_df['model_mape'].mean(),
                        'daily_results': results_df,
                        'all_predictions': all_predictions_df
                    }
                    
                    print(f"\nLoaded {len(actuals_files)} slates, {len(all_predictions_df)} predictions")
                    print(f"Model MAPE: {results['model_mean_mape']:.2f}%")
                    print(f"Improvement: {results['mape_improvement']:+.2f}%")
                    print("\nSkip to cell 10 to visualize results")
else:
    print("Set LOAD_FROM_PREVIOUS = True to load previous results")

Found 62 previous run(s)
Loading from: 20251019_191117
ERROR: No result files found


In [17]:
from src.filters import ColumnFilter, InjuryFilter
from src.filters.player_filters import PlayerIDFilter, PlayerNameFilter, PlayerIDFromCSVFilter

# Build player filters from configuration
player_filters = []

# Salary filters
if FILTER_SALARY_MIN is not None:
    player_filters.append(ColumnFilter('salary', '>=', FILTER_SALARY_MIN))
    print(f'Added filter: salary >= {FILTER_SALARY_MIN}')

if FILTER_SALARY_MAX is not None:
    player_filters.append(ColumnFilter('salary', '<=', FILTER_SALARY_MAX))
    print(f'Added filter: salary <= {FILTER_SALARY_MAX}')

# Injury filters
if FILTER_EXCLUDE_OUT or FILTER_EXCLUDE_DOUBTFUL or FILTER_EXCLUDE_QUESTIONABLE:
    injury_filter = InjuryFilter(
        exclude_out=FILTER_EXCLUDE_OUT,
        exclude_doubtful=FILTER_EXCLUDE_DOUBTFUL,
        exclude_questionable=FILTER_EXCLUDE_QUESTIONABLE
    )
    player_filters.append(injury_filter)
    excluded = []
    if FILTER_EXCLUDE_OUT:
        excluded.append('OUT')
    if FILTER_EXCLUDE_DOUBTFUL:
        excluded.append('DOUBTFUL')
    if FILTER_EXCLUDE_QUESTIONABLE:
        excluded.append('QUESTIONABLE')
    print(f'Added filter: exclude injury status {", ".join(excluded)}')

# Player ID filters
if FILTER_PLAYER_IDS:
    # Parse comma or space-separated player IDs
    if isinstance(FILTER_PLAYER_IDS, str):
        player_ids = [pid.strip() for pid in FILTER_PLAYER_IDS.replace(',', ' ').split() if pid.strip()]
    else:
        player_ids = FILTER_PLAYER_IDS if isinstance(FILTER_PLAYER_IDS, list) else [FILTER_PLAYER_IDS]
    
    if player_ids:
        player_id_filter = PlayerIDFilter(player_ids)
        player_filters.append(player_id_filter)
        ids_display = ', '.join(str(pid) for pid in player_ids[:5])
        if len(player_ids) > 5:
            ids_display += f", ... (+{len(player_ids) - 5} more)"
        print(f'Added filter: player ID in [{ids_display}]')

# Player name filters
if FILTER_PLAYER_NAMES:
    # Parse comma or space-separated player names
    if isinstance(FILTER_PLAYER_NAMES, str):
        player_names = [name.strip() for name in FILTER_PLAYER_NAMES.replace(',', '|').split('|') if name.strip()]
    else:
        player_names = FILTER_PLAYER_NAMES if isinstance(FILTER_PLAYER_NAMES, list) else [FILTER_PLAYER_NAMES]
    
    if player_names:
        player_name_filter = PlayerNameFilter(player_names, case_sensitive=False)
        player_filters.append(player_name_filter)
        names_display = ', '.join(player_names[:3])
        if len(player_names) > 3:
            names_display += f", ... (+{len(player_names) - 3} more)"
        print(f'Added filter: player name contains [{names_display}]')

# Player ID from CSV file
if FILTER_PLAYERS_CSV:
    try:
        csv_filter = PlayerIDFromCSVFilter(FILTER_PLAYERS_CSV)
        player_filters.append(csv_filter)
        print(f'Added filter: player IDs from CSV ({len(csv_filter.player_ids)} players)')
    except FileNotFoundError as e:
        print(f'ERROR: {e}')
    except ValueError as e:
        print(f'ERROR: {e}')

if player_filters:
    print(f'\nTotal filters: {len(player_filters)}')
else:
    print('No player filters configured')

backtest = WalkForwardBacktest(
    db_path=DB_PATH,
    train_start=TRAIN_START,
    train_end=TRAIN_END,
    test_start=TEST_START,
    test_end=TEST_END,
    model_type=MODEL_TYPE,
    model_params=MODEL_PARAMS,
    feature_config=FEATURE_CONFIG,
    output_dir=OUTPUT_DIR,
    per_player_models=PER_PLAYER_MODELS,
    min_player_games=MIN_PLAYER_GAMES,
    min_games_for_benchmark=MIN_GAMES_FOR_BENCHMARK,
    recalibrate_days=RECALIBRATE_DAYS,
    num_seasons=NUM_SEASONS,
    salary_tiers=SALARY_TIERS,
    save_models=SAVE_MODELS,
    save_predictions=SAVE_PREDICTIONS,
    n_jobs=N_JOBS,
    player_filters=player_filters if player_filters else None
)

print('\nRunning backtest...')
results = backtest.run()

if 'error' in results:
    print(f"ERROR: {results['error']}")
else:
    print(f"\nBacktest completed successfully!")
    print(f"Processed {results['num_slates']} slates")
    print(f"Model MAPE: {results['model_mean_mape']:.2f}%")
    print(f"Benchmark MAPE: {results['benchmark_mean_mape']:.2f}%")
    print(f"Improvement: {results['mape_improvement']:+.2f}%")

2025-10-19 20:22:44,311 - src.walk_forward_backtest - WARNING - train_end (20250214) is after test_start (20250205). Training data will overlap with test window.
2025-10-19 20:22:44,312 - src.data.storage.sqlite_storage - INFO - Initialized SQLiteStorage with database: c:\Users\antho\OneDrive\Documents\Repositories\delapan-fantasy\nba_dfs.db
2025-10-19 20:22:44,317 - src.utils.feature_config - INFO - Loaded feature config: Opponent Features
2025-10-19 20:22:44,317 - src.utils.feature_config - INFO - Added RollingStatsTransformer: windows=[3, 10], stats=23, include_std=True
2025-10-19 20:22:44,317 - src.utils.feature_config - INFO - Added OpponentStatsTransformer: 12 features, lookback=365d
2025-10-19 20:22:44,318 - src.utils.feature_config - INFO - Added EWMATransformer: span=5, stats=23
2025-10-19 20:22:44,318 - src.utils.feature_config - INFO - Added TargetTransformer: target_col=fpts, shift_periods=-1
2025-10-19 20:22:44,318 - src.utils.feature_config - INFO - Added InjuryTransforme

Added filter: salary >= 5000
Added filter: player name contains [Lebron James, Stephen Curry]

Total filters: 2

Running backtest...

Backtesting 9 slates from 20250205 to 20250215



2025-10-19 20:22:44,460 - src.walk_forward_backtest - INFO - Loaded 17552 training records
2025-10-19 20:22:44,460 - src.walk_forward_backtest - INFO - Building features for benchmark...
2025-10-19 20:22:44,659 - src.walk_forward_backtest - INFO - Calculated fantasy points for training data
Fitting transformers:   0%|          | 0/5 [00:00<?, ?it/s]2025-10-19 20:22:44,661 - src.features.transformers.opponent_stats - INFO - Fitting OpponentStatsTransformer - calculating team statistics
2025-10-19 20:22:45,347 - src.features.transformers.opponent_stats - INFO - Calculated statistics for 30 teams


KeyboardInterrupt: 

## Extract Results

Extract the daily results and predictions dataframes from the backtest results.

In [ ]:
if 'error' not in results:
    results_df = results['daily_results']
    all_predictions_df = results['all_predictions']
    
    print('='*80)
    print('BACKTEST RESULTS SUMMARY')
    print('='*80)
    print(f'\nNumber of Slates: {results["num_slates"]}')
    print(f'Date Range: {results["date_range"]}')
    print(f'\nTotal Players Evaluated: {results["total_players_evaluated"]:.0f}')
    print(f'Average Players per Slate: {results["avg_players_per_slate"]:.1f}')
    print(f'\nModel Performance:')
    print(f'  Mean MAPE: {results["model_mean_mape"]:.2f}%')
    print(f'  Median MAPE: {results["model_median_mape"]:.2f}%')
    print(f'  Std MAPE: {results["model_std_mape"]:.2f}%')
    print(f'  Mean RMSE: {results["model_mean_rmse"]:.2f}')
    print(f'  Mean MAE: {results["model_mean_mae"]:.2f}')
    print(f'  Mean Correlation: {results["model_mean_correlation"]:.3f}')
    print(f'\nBenchmark Performance:')
    print(f'  Mean MAPE: {results["benchmark_mean_mape"]:.2f}%')
    print(f'  Median MAPE: {results["benchmark_median_mape"]:.2f}%')
    print(f'\nImprovement (Model vs Benchmark):')
    print(f'  MAPE Improvement: {results["mape_improvement"]:+.2f}%')
    
    if 'statistical_test' in results:
        print(f'\nStatistical Significance:')
        print(f'  p-value: {results["statistical_test"]["p_value"]:.6f}')
        print(f'  Cohen\'s d: {results["statistical_test"]["cohens_d"]:.4f}')
        print(f'  Effect size: {results["statistical_test"]["effect_size"]}')

NameError: name 'results' is not defined

## Visualizations

Interactive Altair visualizations showing backtest performance metrics.

In [ ]:
from src.evaluation.altair_visualizations import AltairVisualizer

# Initialize visualizer with dark theme
viz = AltairVisualizer(output_dir=repo_root / 'data' / 'outputs', theme='dark')

# Create performance metrics dashboard
# Prepare data - add slate_number for better x-axis
results_df_viz = results_df.copy()
results_df_viz['slate_number'] = range(len(results_df_viz))

# MAPE over time (model vs benchmark)
mape_data = results_df_viz[['slate_number', 'date', 'model_mape', 'benchmark_mape']]
mape_chart = viz.create_multi_line_chart(
    data=mape_data,
    x='date',
    y_columns=['model_mape', 'benchmark_mape'],
    title='MAPE Over Time - Model vs Benchmark',
    width=450,
    height=250
)

# RMSE over time
rmse_chart = viz.create_line_chart(
    data=results_df_viz,
    x='date',
    y='model_rmse',
    title='RMSE Over Time',
    width=450,
    height=250
)

# Correlation over time
corr_chart = viz.create_line_chart(
    data=results_df_viz,
    x='date',
    y='model_corr',
    title='Correlation Over Time',
    width=450,
    height=250
)

# Players evaluated per slate
players_chart = viz.create_bar_chart(
    data=results_df_viz,
    x='date',
    y='num_players',
    title='Players Evaluated Per Slate',
    width=450,
    height=250
)

# Combine into 2x2 dashboard
dashboard = viz.create_dashboard([mape_chart, rmse_chart, corr_chart, players_chart], columns=2)
dashboard

In [ ]:
if 'error' not in results and 'tier_comparison' in results:
    tier_comparison = results['tier_comparison']
    
    # MAPE by salary tier (grouped bar)
    tier_data = tier_comparison[['salary_tier', 'model_mape', 'benchmark_mape']].melt(
        id_vars='salary_tier',
        value_vars=['model_mape', 'benchmark_mape'],
        var_name='metric',
        value_name='mape'
    )
    
    tier_mape_chart = viz.create_bar_chart(
        data=tier_data,
        x='salary_tier',
        y='mape',
        color='metric',
        title='MAPE by Salary Tier',
        width=400,
        height=350
    )
    
    # Improvement chart
    improvement_chart = viz.create_bar_chart(
        data=tier_comparison,
        x='salary_tier',
        y='mape_improvement',
        color='mape_improvement',
        title='Model Improvement Over Benchmark',
        width=400,
        height=350
    )
    
    # Combine
    tier_dashboard = viz.create_dashboard([tier_mape_chart, improvement_chart], columns=2)
    tier_dashboard
else:
    print('Tier comparison not available in results')

In [ ]:
if 'error' not in results and 'benchmark_comparison' in results:
    comparison_results = results['benchmark_comparison']
    print(comparison_results['summary'])
else:
    print('Benchmark comparison not available')

Benchmark comparison not available


In [ ]:
if 'error' not in results:
    # Use built-in backtest dashboard method
    backtest_dashboard = viz.create_backtest_dashboard(results)
    backtest_dashboard

In [ ]:
if 'error' not in results and not all_predictions_df.empty:
    # Filter for valid comparisons
    comparison_df = all_predictions_df[
        (all_predictions_df['projected_fpts'] > 0) & 
        (all_predictions_df['benchmark_pred'] > 0)
    ].copy()
    
    # Calculate errors
    comparison_df['model_error'] = np.abs(comparison_df['projected_fpts'] - comparison_df['actual_fpts'])
    comparison_df['benchmark_error'] = np.abs(comparison_df['benchmark_pred'] - comparison_df['actual_fpts'])
    comparison_df['error_diff'] = comparison_df['benchmark_error'] - comparison_df['model_error']
    
    # Model vs Benchmark error scatter
    error_scatter = viz.create_scatter_plot(
        data=comparison_df,
        x='benchmark_error',
        y='model_error',
        title='Model vs Benchmark Error Comparison',
        width=500,
        height=400
    )
    
    # Error difference histogram
    error_hist = viz.create_histogram(
        data=comparison_df,
        column='error_diff',
        bins=30,
        title='Error Difference Distribution (Positive = Model Better)',
        width=500,
        height=400,
        color='#18FF6D'
    )
    
    # Combine
    error_dashboard = viz.create_dashboard([error_scatter, error_hist], columns=2)
    error_dashboard

In [ ]:
if 'error' not in results and not all_predictions_df.empty:
    # Filter for valid comparisons
    comparison_df = all_predictions_df[
        (all_predictions_df['projected_fpts'] > 0) & 
        (all_predictions_df['benchmark_pred'] > 0)
    ].copy()
    
    # Dark theme and vibrant colors
    vibrant_colors = {
        "scatter": "#00D7FF",        # Cyan
        "diagonal": "#FF0080",       # Pink/magenta
        "histogram": "#18FF6D",      # Vibrant green
        "histogram_border": "#22272e",
        "vline_zero": "#FF0080",     # Pink/magenta
        "vline_mean": "#FFD700",     # Gold/yellow
    }
    
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            '<span style="color:white"><b>Model vs Benchmark Error Comparison</b></span>', 
            '<span style="color:white"><b>Error Difference Distribution</b><br>(Positive = Model Better)</span>'
        ),
        horizontal_spacing=0.15
    )
    
    # Calculate errors
    comparison_df['model_error'] = np.abs(comparison_df['projected_fpts'] - comparison_df['actual_fpts'])
    comparison_df['benchmark_error'] = np.abs(comparison_df['benchmark_pred'] - comparison_df['actual_fpts'])
    
    # Scatter plot: Model vs Benchmark Error
    fig.add_trace(
        go.Scatter(
            x=comparison_df['benchmark_error'], y=comparison_df['model_error'],
            mode='markers',
            marker=dict(size=7, opacity=0.7, color=vibrant_colors["scatter"], line=dict(width=0)),
            name='Errors',
            showlegend=True
        ),
        row=1, col=1
    )
    
    # Add diagonal line for equal error
    max_error = max(comparison_df['benchmark_error'].max(), comparison_df['model_error'].max()) * 1.03
    fig.add_trace(
        go.Scatter(
            x=[0, max_error], y=[0, max_error],
            mode='lines',
            line=dict(color=vibrant_colors["diagonal"], dash='dash', width=2),
            name='Equal error',
            showlegend=True
        ),
        row=1, col=1
    )
    
    # Histogram of error differences
    error_diff = comparison_df['benchmark_error'] - comparison_df['model_error']
    fig.add_trace(
        go.Histogram(
            x=error_diff, nbinsx=30,
            marker=dict(color=vibrant_colors["histogram"], opacity=0.85, 
                        line=dict(color=vibrant_colors["histogram_border"], width=1.2)),
            name='Error Difference',
            showlegend=True
        ),
        row=1, col=2
    )
    
    # Add vertical lines for reference
    fig.add_vline(
        x=0,
        line_dash="dash",
        line_color=vibrant_colors["vline_zero"],
        row=1, col=2,
        annotation_text="<b style='color:#FF0080'>No difference</b>",
        annotation_position="top"
    )
    fig.add_vline(
        x=error_diff.mean(),
        line_dash="dash",
        line_color=vibrant_colors["vline_mean"],
        row=1, col=2,
        annotation_text=f"<b style='color:#FFD700'>Mean: {error_diff.mean():.2f}</b>",
        annotation_position="top right"
    )
    
    # Update layout for dark theme
    axis_style = dict(color="white", showline=True, linewidth=1.7, linecolor='#666', zerolinecolor="#444")
    fig.update_xaxes(title_text="<b style='color:#75eaff'>Benchmark Error</b>", row=1, col=1, tickfont_color="white", **axis_style)
    fig.update_yaxes(title_text="<b style='color:#FF0080'>Model Error</b>", row=1, col=1, tickfont_color="white", **axis_style)
    fig.update_xaxes(title_text="<b style='color:#18FF6D'>Error Difference (Benchmark - Model)</b>", row=1, col=2, tickfont_color="white", **axis_style)
    fig.update_yaxes(title_text="<b style='color:#FFD700'>Frequency</b>", row=1, col=2, tickfont_color="white", **axis_style)
    
    fig.update_layout(
        height=800,
        width=1400,
        showlegend=True,
        legend=dict(
            orientation="h",
            y=1.05,
            yanchor="bottom",
            xanchor="center",
            x=0.5,
            font=dict(color="white", size=13)
        ),
        plot_bgcolor="#22272e",
        paper_bgcolor="#22272e",
        font=dict(family="Segoe UI, Roboto, Arial", size=15, color="white"),
        title_text="<b>Model vs Benchmark Error Analysis</b>",
        title_x=0.5,
        margin=dict(l=60, r=60, t=80, b=55)
    )
    
    fig.show()